# Remote File Workflow Probe

Date: 2026-06-15

This notebook simulates the Lambda remote-agent finding locally without using paid cloud:

- `--file -` with closed stdin should fail loudly before adding a cell.
- Running cells from real source files should succeed.
- Command output should give enough status, stdout/stderr, and artifact information for a local agent to iterate.



In [1]:
from pathlib import Path
import json
import sys

artifact_dir = Path("artifacts/agent_rounds/20260615_round10/remote_file_workflow")
artifact_dir.mkdir(parents=True, exist_ok=True)

metric = {
    "round": "20260615_round10",
    "workflow": "source-file",
    "value": 41,
    "python": sys.version.split()[0],
    "cwd": str(Path.cwd()),
}

artifact_path = artifact_dir / "remote_metric.json"
artifact_path.write_text(json.dumps(metric, indent=2, sort_keys=True) + "\n", encoding="utf-8")

remote_value = metric["value"]

print("source_file_workflow ok")
print(f"remote_value {remote_value}")
print(f"artifact {artifact_path}")



source_file_workflow ok
remote_value 41
artifact artifacts/agent_rounds/20260615_round10/remote_file_workflow/remote_metric.json


In [2]:
from pathlib import Path
import json

artifact_dir = Path("artifacts/agent_rounds/20260615_round10/remote_file_workflow")
artifact_path = artifact_dir / "remote_metric.json"
metric = json.loads(artifact_path.read_text(encoding="utf-8"))

remote_value_plus_one = remote_value + 1
metric["value_plus_one"] = remote_value_plus_one

derived_path = artifact_dir / "remote_metric_plus_one.json"
derived_path.write_text(json.dumps(metric, indent=2, sort_keys=True) + "\n", encoding="utf-8")

print("reused_live_kernel_state ok")
print(f"remote_value_plus_one {remote_value_plus_one}")
print(f"artifact {derived_path}")



reused_live_kernel_state ok
remote_value_plus_one 42
artifact artifacts/agent_rounds/20260615_round10/remote_file_workflow/remote_metric_plus_one.json


# Command Log

Working directory:

```sh
/Users/guraltoo/Documents/dev/proj/experiments/notebook-lens
```

## Notebook Setup

```sh
uv run notebook-lens new notebooks/agent_rounds/20260615_round10/remote_file_workflow.ipynb
```

Exit code: `0`

```text
notebook: agent_rounds/20260615_round10/remote_file_workflow.ipynb
input_path: notebooks/agent_rounds/20260615_round10/remote_file_workflow.ipynb
absolute_path: /Users/guraltoo/Documents/dev/proj/experiments/notebook-lens/notebooks/agent_rounds/20260615_round10/remote_file_workflow.ipynb
status: created
```

```sh
uv run notebook-lens add-markdown notebooks/agent_rounds/20260615_round10/remote_file_workflow.ipynb --desc "Probe context" --file tmp/agent_rounds/20260615_round10/remote_file_workflow/00_context.md
```

Exit code: `0`

```text
notebook: agent_rounds/20260615_round10/remote_file_workflow.ipynb
input_path: notebooks/agent_rounds/20260615_round10/remote_file_workflow.ipynb
absolute_path: /Users/guraltoo/Documents/dev/proj/experiments/notebook-lens/notebooks/agent_rounds/20260615_round10/remote_file_workflow.ipynb
cell_index: 0
cell_id: a383572e
status: added
```

## Empty Closed Stdin

```sh
uv run notebook-lens run-cell notebooks/agent_rounds/20260615_round10/remote_file_workflow.ipynb --desc "Closed stdin failure probe" --file - < /dev/null
```

Exit code: `2`

```text
error: code source is empty; pass a real --file path. When running through remote SSH wrappers, do not rely on --file - unless stdin is known to be forwarded.
```

```sh
uv run notebook-lens state notebooks/agent_rounds/20260615_round10/remote_file_workflow.ipynb --outputs none --max-tokens 1000
```

Exit code: `0`

```text
## /Users/guraltoo/Documents/dev/proj/experiments/notebook-lens/notebooks/agent_rounds/20260615_round10/remote_file_workflow.ipynb

[0] ok markdown - Probe context
cell_id: a383572e
```

## Source-File Workflow

```sh
uv run notebook-lens run-cell notebooks/agent_rounds/20260615_round10/remote_file_workflow.ipynb --desc "Set remote-style state from source file" --file tmp/agent_rounds/20260615_round10/remote_file_workflow/01_set_remote_state.py
```

Exit code: `0`

```text
notebook: agent_rounds/20260615_round10/remote_file_workflow.ipynb
input_path: notebooks/agent_rounds/20260615_round10/remote_file_workflow.ipynb
absolute_path: /Users/guraltoo/Documents/dev/proj/experiments/notebook-lens/notebooks/agent_rounds/20260615_round10/remote_file_workflow.ipynb
cell_index: 1
cell_id: 05773dce
status: ok
stdout:
source_file_workflow ok
remote_value 41
artifact artifacts/agent_rounds/20260615_round10/remote_file_workflow/remote_metric.json
artifact_scope:
- artifacts/agent_rounds/20260615_round10/remote_file_workflow (inferred)
artifacts:
- artifacts/agent_rounds/20260615_round10/remote_file_workflow/remote_metric.json (173 bytes, new)
```

```sh
uv run notebook-lens run-cell notebooks/agent_rounds/20260615_round10/remote_file_workflow.ipynb --desc "Reuse remote-style kernel state from source file" --file tmp/agent_rounds/20260615_round10/remote_file_workflow/02_reuse_remote_state.py
```

Exit code: `0`

```text
notebook: agent_rounds/20260615_round10/remote_file_workflow.ipynb
input_path: notebooks/agent_rounds/20260615_round10/remote_file_workflow.ipynb
absolute_path: /Users/guraltoo/Documents/dev/proj/experiments/notebook-lens/notebooks/agent_rounds/20260615_round10/remote_file_workflow.ipynb
cell_index: 2
cell_id: 03eb4ab8
status: ok
stdout:
reused_live_kernel_state ok
remote_value_plus_one 42
artifact artifacts/agent_rounds/20260615_round10/remote_file_workflow/remote_metric_plus_one.json
artifact_scope:
- artifacts/agent_rounds/20260615_round10/remote_file_workflow (inferred)
artifacts:
- artifacts/agent_rounds/20260615_round10/remote_file_workflow/remote_metric_plus_one.json (197 bytes, new)
```

## Agent Inspection

```sh
uv run notebook-lens state notebooks/agent_rounds/20260615_round10/remote_file_workflow.ipynb --outputs summary --max-tokens 2000
```

Exit code: `0`

Result: rendered the context markdown, both code cells, and stdout excerpts including `remote_value 41` and `remote_value_plus_one 42`.

```sh
uv run notebook-lens show-cell notebooks/agent_rounds/20260615_round10/remote_file_workflow.ipynb 2 --outputs full
```

Exit code: `0`

Result: rendered the second cell source and full stdout, including `reused_live_kernel_state ok`.

```sh
uv run notebook-lens run-clean notebooks/agent_rounds/20260615_round10/remote_file_workflow.ipynb
```

Exit code: `0`

```text
status: ok
executed_cells: 2
total_code_cells: 2
cell_outputs:
- cell_index: 1, cell_id: 05773dce, status: ok
  stdout_excerpt:
    source_file_workflow ok
    remote_value 41
    artifact artifacts/agent_rounds/20260615_round10/remote_file_workflow/remote_metric.json
- cell_index: 2, cell_id: 03eb4ab8, status: ok
  stdout_excerpt:
    reused_live_kernel_state ok
    remote_value_plus_one 42
    artifact artifacts/agent_rounds/20260615_round10/remote_file_workflow/remote_metric_plus_one.json
artifact_note: 2 preexisting files in scope; run-clean does not delete artifacts.
artifact_scope:
- artifacts/agent_rounds/20260615_round10/remote_file_workflow (inferred)
artifacts:
- artifacts/agent_rounds/20260615_round10/remote_file_workflow/remote_metric.json (173 bytes, modified)
- artifacts/agent_rounds/20260615_round10/remote_file_workflow/remote_metric_plus_one.json (197 bytes, modified)
```

## Final Kernel Reset

```sh
uv run notebook-lens reset notebooks/agent_rounds/20260615_round10/remote_file_workflow.ipynb
```

Exit code: `0`

```text
notebook: agent_rounds/20260615_round10/remote_file_workflow.ipynb
input_path: notebooks/agent_rounds/20260615_round10/remote_file_workflow.ipynb
absolute_path: /Users/guraltoo/Documents/dev/proj/experiments/notebook-lens/notebooks/agent_rounds/20260615_round10/remote_file_workflow.ipynb
status: reset
```
